# Análisis del conjunto meteorológico ERA5

En este cuaderno se revisa el archivo meteorológico de ERA5 descargado de Open-Meteo para Sídney, que cubre de julio de 2010 a junio de 2013. Se describe cómo está organizado, qué variables contiene y en qué unidades, y se comprueba si tiene duplicados, horas ausentes, celdas vacías o valores fuera de rango. Al final se recogen los cambios que hay que aplicarle antes de unirlo con los datos de Ausgrid.

El archivo original solo se lee. Las transformaciones se harán en los scripts de `src/preprocessing`.

## 1. Carga del archivo y control de versión

El archivo es una versión congelada y su huella SHA-256 está registrada en `docs/data/MANIFEST.md`. Se comprueba la huella antes del análisis para asegurar que se trabaja con la misma versión que el resto del equipo.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

weather_path = project_root / "data" / "weather" / "era5_sydney_2010_2013.parquet"
if not weather_path.is_file():
    raise FileNotFoundError(f"No se encontró el archivo: {weather_path}")

# Huella registrada en docs/data/MANIFEST.md
expected_sha256 = "63e45ef341654d177c3670d109d54d344e971859fd46b799b1b7bb0133ae0f58"
file_sha256 = hashlib.sha256(weather_path.read_bytes()).hexdigest()

weather = pd.read_parquet(weather_path)
print("Huella SHA-256 coincide con el manifiesto:", file_sha256 == expected_sha256)
print("Filas:", len(weather))
print("Columnas:", len(weather.columns))
print("Índice:", weather.index.name, "| zona horaria:", weather.index.tz)
weather.head()

Huella SHA-256 coincide con el manifiesto: True
Filas: 26304
Columnas: 9
Índice: time | zona horaria: UTC


,temperature_2m,relative_humidity_2m,precipitation,cloud_cover,shortwave_radiation,direct_normal_irradiance,diffuse_radiation,wind_speed_10m,surface_pressure
time,,,,,,,,,
2010-07-01 00:00:00+00:00,7.5,77,0.0,35,320.0,526.8,116.0,5.1,1014.6
2010-07-01 01:00:00+00:00,11.9,59,0.0,10,446.0,698.7,106.0,5.5,1014.1
2010-07-01 02:00:00+00:00,14.4,49,0.0,8,518.0,782.5,97.0,5.8,1013.6
2010-07-01 03:00:00+00:00,14.7,48,0.0,2,527.0,834.8,78.0,5.0,1012.2
2010-07-01 04:00:00+00:00,15.3,46,0.0,9,461.0,798.0,73.0,4.4,1012.2


**Resultado.** La huella coincide con la registrada, así que el archivo es la versión congelada del proyecto. Tiene 26.304 filas y 9 columnas. Cada fila corresponde a una hora y cada columna a una variable meteorológica. La marca temporal no es una columna: está en el índice `time`, en UTC.

## 2. Variables y unidades

Open-Meteo no entrega todas las variables de la misma forma. Unas son valores instantáneos tomados en la hora indicada y otras son promedios o acumulados de la hora anterior. Esta diferencia se comprueba en la sección 7 y define cómo se une ERA5 con Ausgrid.

In [2]:
variable_info = pd.DataFrame(
    [
        ("temperature_2m", "Temperatura del aire a 2 m", "°C", "instantáneo"),
        ("relative_humidity_2m", "Humedad relativa a 2 m", "%", "instantáneo"),
        ("precipitation", "Precipitación", "mm", "acumulado de la hora anterior"),
        ("cloud_cover", "Cobertura nubosa total", "%", "instantáneo"),
        ("shortwave_radiation", "Irradiancia global horizontal (GHI)", "W/m²", "promedio de la hora anterior"),
        ("direct_normal_irradiance", "Irradiancia directa normal (DNI)", "W/m²", "promedio de la hora anterior"),
        ("diffuse_radiation", "Irradiancia difusa horizontal (DHI)", "W/m²", "promedio de la hora anterior"),
        ("wind_speed_10m", "Velocidad del viento a 10 m", "km/h", "instantáneo"),
        ("surface_pressure", "Presión en superficie", "hPa", "instantáneo"),
    ],
    columns=["variable", "descripción", "unidad", "tipo de valor"],
).set_index("variable")
variable_info["tipo de dato"] = weather.dtypes.astype(str)
variable_info

,descripción,unidad,tipo de valor,tipo de dato
variable,,,,
temperature_2m,Temperatura del aire a 2 m,°C,instantáneo,float64
relative_humidity_2m,Humedad relativa a 2 m,%,instantáneo,int64
precipitation,Precipitación,mm,acumulado de la hora anterior,float64
cloud_cover,Cobertura nubosa total,%,instantáneo,int64
shortwave_radiation,Irradiancia global horizontal (GHI),W/m²,promedio de la hora anterior,float64
direct_normal_irradiance,Irradiancia directa normal (DNI),W/m²,promedio de la hora anterior,float64
diffuse_radiation,Irradiancia difusa horizontal (DHI),W/m²,promedio de la hora anterior,float64
wind_speed_10m,Velocidad del viento a 10 m,km/h,instantáneo,float64
surface_pressure,Presión en superficie,hPa,instantáneo,float64


**Resultado.** Las nueve variables son numéricas: `relative_humidity_2m` y `cloud_cover` se guardan como enteros y el resto como decimales. No hay columnas de texto ni fechas que convertir. La única unidad que hay que cambiar es la del viento: está en km/h y pvlib, que se usará para estimar la generación fotovoltaica, trabaja en m/s.

## 3. Continuidad temporal y duplicados

El índice temporal de ERA5 debe avanzar de una hora en una hora. Se comprueban marcas duplicadas, el orden del índice, horas ausentes y celdas vacías antes de usar estos datos en otros análisis.

In [3]:
weather_index = weather.index
expected_index = pd.date_range(
    weather_index.min(), weather_index.max(), freq="h", tz=weather_index.tz
)
missing_times = expected_index.difference(weather_index)

weather_quality = pd.DataFrame(
    {"resultado": [
        str(weather_index.min()),
        str(weather_index.max()),
        str(pd.infer_freq(weather_index)),
        bool(weather_index.is_monotonic_increasing),
        int(weather_index.duplicated().sum()),
        len(missing_times),
        int(weather.isna().sum().sum()),
    ]},
    index=[
        "Primera marca temporal",
        "Última marca temporal",
        "Frecuencia inferida",
        "Índice ordenado",
        "Marcas duplicadas",
        "Horas ausentes",
        "Celdas vacías",
    ],
)
weather_quality

,resultado
Primera marca temporal,2010-07-01 00:00:00+00:00
Última marca temporal,2013-06-30 23:00:00+00:00
Frecuencia inferida,h
Índice ordenado,True
Marcas duplicadas,0
Horas ausentes,0
Celdas vacías,0


**Resultado.** El período va del 1 de julio de 2010 a las 00:00 UTC al 30 de junio de 2013 a las 23:00 UTC. El índice está ordenado, la frecuencia es horaria y no hay marcas duplicadas, horas ausentes ni celdas vacías. El archivo no necesita deduplicación ni imputación.

## 4. Rangos de valores

Se resumen el mínimo, la mediana, la media y el máximo de cada variable para comprobar que los valores son coherentes con sus unidades.

In [4]:
weather_profile = pd.DataFrame({
    "unidad": variable_info["unidad"],
    "mínimo": weather.min(),
    "mediana": weather.median(),
    "media": weather.mean().round(1),
    "máximo": weather.max(),
})
weather_profile

,unidad,mínimo,mediana,media,máximo
temperature_2m,°C,1.3,17.0,17.0,44.8
relative_humidity_2m,%,10.0,76.0,72.5,100.0
precipitation,mm,0.0,0.0,0.1,9.0
cloud_cover,%,0.0,49.0,50.3,100.0
shortwave_radiation,W/m²,0.0,7.0,187.3,1111.0
direct_normal_irradiance,W/m²,0.0,0.0,218.7,1031.8
diffuse_radiation,W/m²,0.0,5.0,61.2,468.0
wind_speed_10m,km/h,0.0,9.4,10.5,39.5
surface_pressure,hPa,978.4,1007.2,1007.3,1028.0


**Resultado.** Los rangos corresponden al clima de Sídney: temperatura entre 1,3 y 44,8 °C, humedad entre 10 y 100 % y presión entre 978 y 1028 hPa. La mediana de la precipitación es 0 porque la mayoría de las horas no llueve. La mediana de la radiación global es de solo 7 W/m² porque cerca de la mitad de las horas son nocturnas y valen 0. Esos ceros son valores correctos y no datos faltantes.

## 5. Comprobación de dominios físicos

Se cuentan los valores que quedan fuera de los límites físicos de cada variable: porcentajes entre 0 y 100, precipitación, radiación, viento y presión no negativos, y temperatura dentro de un intervalo amplio. Los valores fuera de dominio solo se marcan; no se eliminan.

In [5]:
domain_checks = {
    "temperature_2m": ("-50 a 60 °C", ~weather["temperature_2m"].between(-50, 60)),
    "relative_humidity_2m": ("0 a 100 %", ~weather["relative_humidity_2m"].between(0, 100)),
    "precipitation": (">= 0 mm", weather["precipitation"] < 0),
    "cloud_cover": ("0 a 100 %", ~weather["cloud_cover"].between(0, 100)),
    "shortwave_radiation": (">= 0 W/m²", weather["shortwave_radiation"] < 0),
    "direct_normal_irradiance": (">= 0 W/m²", weather["direct_normal_irradiance"] < 0),
    "diffuse_radiation": (">= 0 W/m²", weather["diffuse_radiation"] < 0),
    "wind_speed_10m": (">= 0 km/h", weather["wind_speed_10m"] < 0),
    "surface_pressure": (">= 0 hPa", weather["surface_pressure"] < 0),
}
domain_summary = pd.DataFrame([
    {"variable": variable, "dominio": domain, "fuera de dominio": int(mask.sum())}
    for variable, (domain, mask) in domain_checks.items()
]).set_index("variable")
domain_summary

,dominio,fuera de dominio
variable,,
temperature_2m,-50 a 60 °C,0
relative_humidity_2m,0 a 100 %,0
precipitation,>= 0 mm,0
cloud_cover,0 a 100 %,0
shortwave_radiation,>= 0 W/m²,0
direct_normal_irradiance,>= 0 W/m²,0
diffuse_radiation,>= 0 W/m²,0
wind_speed_10m,>= 0 km/h,0
surface_pressure,>= 0 hPa,0


**Resultado.** Ningún valor queda fuera de dominio en las 26.304 horas. El intervalo de temperatura es solo un filtro amplio para detectar errores groseros, como un cambio de unidades; los extremos se revisan en la sección siguiente.

## 6. Valores extremos

Para cada variable se compara el máximo con el percentil 99 y se registra la hora en que ocurre, en UTC y en hora local de Sídney. Un máximo muy por encima del p99 se revisa antes de decidir si es un error.

In [6]:
extreme_rows = []
for variable in weather.columns:
    series = weather[variable]
    maximum_time = series.idxmax()
    p99 = float(series.quantile(0.99))
    maximum = float(series.loc[maximum_time])
    extreme_rows.append({
        "variable": variable,
        "p99": round(p99, 1),
        "máximo": maximum,
        "máximo / p99": round(maximum / p99, 2),
        "hora UTC": maximum_time.strftime("%Y-%m-%d %H:%M"),
        "hora local": maximum_time.tz_convert("Australia/Sydney").strftime("%Y-%m-%d %H:%M"),
    })
extreme_summary = pd.DataFrame(extreme_rows).set_index("variable")
extreme_summary

,p99,máximo,máximo / p99,hora UTC,hora local
variable,,,,,
temperature_2m,31.4,44.8,1.43,2013-01-18 04:00,2013-01-18 15:00
relative_humidity_2m,98.0,100.0,1.02,2010-11-06 21:00,2010-11-07 08:00
precipitation,1.7,9.0,5.29,2010-11-01 17:00,2010-11-02 04:00
cloud_cover,100.0,100.0,1.00,2010-07-01 14:00,2010-07-02 00:00
shortwave_radiation,949.0,1111.0,1.17,2010-12-20 03:00,2010-12-20 14:00
direct_normal_irradiance,941.4,1031.8,1.10,2012-01-11 02:00,2012-01-11 13:00
diffuse_radiation,334.0,468.0,1.40,2012-11-21 03:00,2012-11-21 14:00
wind_speed_10m,26.4,39.5,1.50,2012-08-10 02:00,2012-08-10 12:00
surface_pressure,1022.3,1028.0,1.01,2011-08-22 00:00,2011-08-22 10:00


**Resultado.** Casi todos los máximos están entre 1,0 y 1,5 veces su p99. Hay dos casos que se revisan aparte:

- **Temperatura de 44,8 °C** el 18 de enero de 2013 a las 15:00 hora local. Ese día Sídney registró su récord histórico, 45,8 °C en Observatory Hill, por lo que el valor es real.
- **Precipitación de 9 mm en una hora**, 5,3 veces el p99. No es un valor anómalo; al contrario, es bajo para una tormenta fuerte. ERA5 representa celdas de unos 31 km y suaviza la lluvia intensa local, así que subestima los picos de precipitación. Es una limitación del reanálisis y no un error del archivo.

La radiación global máxima, 1.111 W/m², ocurre el 20 de diciembre a mediodía, cerca del solsticio de verano, y también es plausible. No se elimina ningún valor.

## 7. Convención temporal de la radiación

En las tres variables de radiación, Open-Meteo asigna a cada marca el promedio de la hora anterior: el valor de las 10:00 es la media entre las 09:00 y las 10:00. Esto se comprueba con la relación física entre las tres componentes:

$$\text{GHI} = \text{DNI} \cdot \cos\theta_z + \text{DHI}$$

donde $\theta_z$ es el ángulo cenital del sol. Si la convención es correcta, la relación se cumple cuando $\theta_z$ se calcula a mitad de la hora anterior ($t - 30$ min) y no en la propia marca $t$. La posición del sol se obtiene con las fórmulas aproximadas de la NOAA. Solo se usan las horas con GHI mayor que 100 W/m².

In [7]:
LATITUDE, LONGITUDE = -33.8688, 151.2093  # punto de la descarga (centro de Sídney)

def cos_zenith(times_utc):
    """Coseno del ángulo cenital solar (fórmulas aproximadas de la NOAA)."""
    day = times_utc.dayofyear.values
    hour = times_utc.hour.values + times_utc.minute.values / 60
    gamma = 2 * np.pi / 365 * (day - 1 + (hour - 12) / 24)
    declination = (0.006918 - 0.399912 * np.cos(gamma) + 0.070257 * np.sin(gamma)
                   - 0.006758 * np.cos(2 * gamma) + 0.000907 * np.sin(2 * gamma)
                   - 0.002697 * np.cos(3 * gamma) + 0.00148 * np.sin(3 * gamma))
    equation_of_time = 229.18 * (0.000075 + 0.001868 * np.cos(gamma) - 0.032077 * np.sin(gamma)
                                 - 0.014615 * np.cos(2 * gamma) - 0.040849 * np.sin(2 * gamma))
    hour_angle = np.radians((hour * 60 + equation_of_time + 4 * LONGITUDE) / 4 - 180)
    lat = np.radians(LATITUDE)
    value = (np.sin(lat) * np.sin(declination)
             + np.cos(lat) * np.cos(declination) * np.cos(hour_angle))
    return np.clip(value, 0, None)

daytime = weather["shortwave_radiation"] > 100
ghi = weather.loc[daytime, "shortwave_radiation"].values
closure_rows = []
for shift_minutes in [0, -30, -60]:
    cz = cos_zenith(weather.index[daytime] + pd.Timedelta(minutes=shift_minutes))
    ghi_estimated = (weather.loc[daytime, "direct_normal_irradiance"].values * cz
                     + weather.loc[daytime, "diffuse_radiation"].values)
    closure_rows.append({
        "sol calculado en": f"t {shift_minutes:+d} min" if shift_minutes else "t",
        "error medio absoluto (W/m²)": round(float(np.abs(ghi_estimated - ghi).mean()), 1),
    })
pd.DataFrame(closure_rows).set_index("sol calculado en")

,error medio absoluto (W/m²)
sol calculado en,
t,29.2
t -30 min,1.4
t -60 min,27.8


**Resultado.** La relación se cumple casi exactamente, con un error de 1,4 W/m², solo cuando el sol se calcula en $t - 30$ min. En $t$ o en $t - 60$ min el error sube a unos 28-29 W/m². Queda confirmado que la radiación de cada marca es el promedio de la hora anterior, y lo mismo se aplica a la precipitación, que es el acumulado de esa hora.

Esto tiene tres consecuencias:

1. Ausgrid, una vez agregado a horas, también etiqueta cada valor con el final de su intervalo. Las dos fuentes se pueden unir directamente por la marca UTC, sin desplazar ninguna.
2. Al calcular la generación con pvlib, la posición del sol se debe evaluar en $t - 30$ min. Esto corresponde a la etapa de modelado de la generación y no al preprocesamiento.
3. La temperatura, la humedad, el viento, la nubosidad y la presión son instantáneas en $t$, es decir, al final de la hora. La diferencia de media hora con el centro del intervalo es pequeña y se deja sin corregir.

## 8. Perfil horario de la radiación

Se comprueba que la radiación sea cero durante la noche y máxima hacia el mediodía. Para esta revisión se usa la hora estándar de Sídney (UTC+10) sin horario de verano: con la hora local, los meses de verano quedarían desplazados una hora y el perfil se mezclaría.

In [8]:
weather_standard = weather.tz_convert("Etc/GMT-10")  # UTC+10 fijo, sin horario de verano
solar_profile = weather_standard.groupby(weather_standard.index.hour).agg(
    ghi_media_W_m2=("shortwave_radiation", "mean"),
    horas_con_radiacion=("shortwave_radiation", lambda values: int((values > 0).sum())),
    horas_totales=("shortwave_radiation", "size"),
).round(1)
solar_profile.index.name = "hora estándar (fin del intervalo)"
solar_profile

,ghi_media_W_m2,horas_con_radiacion,horas_totales
hora estándar (fin del intervalo),,,
0,0.0,0,1096
1,0.0,0,1096
2,0.0,0,1096
3,0.0,0,1096
4,0.0,0,1096
5,0.2,145,1096
6,17.0,499,1096
7,80.5,892,1096
8,204.8,1096,1096


**Resultado.** La radiación vale cero en las marcas de las 20:00 a las 04:00 en todos los días; en las de las 05:00 y las 19:00 solo aparece en verano, cuando el día es más largo. El máximo medio está en la marca de las 12:00, que corresponde al intervalo de 11:00 a 12:00, y la marca de las 13:00 tiene un valor casi igual. Es lo esperado: el mediodía solar en Sídney cae hacia las 11:55 en hora estándar. El perfil no muestra desplazamientos.

## 9. Cobertura respecto a Ausgrid

Ausgrid registra intervalos de 30 minutos en hora local de Sídney desde el 1 de julio de 2010 a las 00:00 hasta el 1 de julio de 2013 a las 00:00. Se calculan las horas de Ausgrid en UTC, etiquetadas con el final de cada hora, y se comparan con las marcas de ERA5.

In [9]:
ausgrid_start_local = pd.Timestamp("2010-07-01 01:00", tz="Australia/Sydney")  # fin de la primera hora
ausgrid_end_local = pd.Timestamp("2013-07-01 00:00", tz="Australia/Sydney")    # fin de la última hora
ausgrid_hours_utc = pd.date_range(
    ausgrid_start_local.tz_convert("UTC"), ausgrid_end_local.tz_convert("UTC"), freq="h"
)

hours_without_weather = ausgrid_hours_utc.difference(weather.index)
weather_hours_unused = weather.index.difference(ausgrid_hours_utc)

print("Horas de Ausgrid en UTC:", ausgrid_hours_utc.min(), "->", ausgrid_hours_utc.max())
print("Horas de Ausgrid sin dato de ERA5:", len(hours_without_weather),
      f"({100 * len(hours_without_weather) / len(ausgrid_hours_utc):.3f} %)")
print("   desde", hours_without_weather.min(), "hasta", hours_without_weather.max())
print("Horas de ERA5 fuera del período de Ausgrid:", len(weather_hours_unused))
print("   desde", weather_hours_unused.min(), "hasta", weather_hours_unused.max())

Horas de Ausgrid en UTC: 2010-06-30 15:00:00+00:00 -> 2013-06-30 14:00:00+00:00
Horas de Ausgrid sin dato de ERA5: 9 (0.034 %)
   desde 2010-06-30 15:00:00+00:00 hasta 2010-06-30 23:00:00+00:00
Horas de ERA5 fuera del período de Ausgrid: 9
   desde 2013-06-30 15:00:00+00:00 hasta 2013-06-30 23:00:00+00:00


**Resultado.** La descarga de ERA5 se hizo por fechas UTC, mientras que Ausgrid usa fechas locales, que en julio van 10 horas por delante. Por eso los dos períodos están desplazados:

- Las primeras 9 horas de Ausgrid, de las 00:00 a las 09:00 hora local del 1 de julio de 2010, no tienen dato de ERA5. Representan el 0,034 % de las horas.
- Las últimas 9 horas de ERA5, del 30 de junio de 2013 de 15:00 a 23:00 UTC, quedan fuera del período de Ausgrid.

Las 9 horas iniciales se eliminarán del conjunto final y las 9 sobrantes de ERA5 se recortarán. La otra opción, volver a descargar ERA5 desde el 30 de junio de 2010, obligaría a cambiar el archivo congelado y el manifiesto para recuperar solo 9 horas, por lo que se descarta.

## 10. Resumen de hallazgos

| Aspecto | Resultado | ¿Requiere cambio? |
|---|---|---|
| Versión del archivo | La huella SHA-256 coincide con el manifiesto | No |
| Estructura | 26.304 filas horarias, 9 variables numéricas, índice `time` en UTC | No |
| Duplicados | 0 marcas duplicadas | No |
| Horas ausentes y celdas vacías | 0 y 0 | No |
| Dominios físicos | 0 valores fuera de dominio | No |
| Extremos | Todos plausibles; la temperatura máxima coincide con el récord de 2013 | No |
| Unidades | Todas correctas; el viento está en km/h | Sí: pasar el viento a m/s |
| Convención temporal | Radiación y precipitación referidas a la hora anterior; el resto, instantáneas | No: coincide con Ausgrid etiquetado al final de la hora |
| Cobertura frente a Ausgrid | 9 horas iniciales de Ausgrid sin clima y 9 horas finales de ERA5 sobrantes | Sí: recortar ERA5 y eliminar esas 9 horas en la unión |

El archivo ERA5 está completo y limpio. No necesita imputación, deduplicación ni eliminación de valores. Solo requiere un cambio de unidad, un recorte del período y la unión con Ausgrid.

## 11. Plan de preprocesamiento de ERA5

Las transformaciones se programarán en `src/preprocessing/`. Para ERA5 se crearán las siguientes funciones en un módulo `weather.py`:

| Función | Qué hace | Motivo |
|---|---|---|
| `load_weather(path)` | Lee el parquet, comprueba la huella SHA-256 contra el manifiesto y que estén las 9 columnas esperadas | Trabajar siempre con la versión congelada |
| `validate_weather(df)` | Comprueba que el índice esté en UTC, ordenado, sin duplicados ni horas ausentes, sin celdas vacías y dentro de los dominios físicos. Si algo falla, detiene el proceso | Repetir automáticamente los controles de este cuaderno |
| `convert_wind_to_ms(df)` | Divide `wind_speed_10m` entre 3,6 y renombra la columna como `wind_speed_10m_ms` | pvlib trabaja en m/s |
| `trim_to_period(df, start_utc, end_utc)` | Conserva solo las horas dentro del período de Ausgrid en UTC | Quitar las 9 horas finales que sobran |
| `prepare_weather(path)` | Aplica las cuatro funciones anteriores en orden y vuelve a validar el resultado | Obtener ERA5 listo para la unión con una sola llamada |

La unión con Ausgrid se hará en un módulo aparte, `merge.py`, con una sola función:

| Función | Qué hace | Motivo |
|---|---|---|
| `merge_ausgrid_weather(ausgrid_hourly, weather)` | Une la tabla horaria de Ausgrid con ERA5 por la marca UTC del final de la hora, con unión izquierda y relación muchos a uno (300 clientes por cada hora de ERA5). Elimina las 9 horas iniciales sin clima y comprueba que el número de filas no cambie salvo por esas horas y que no queden celdas meteorológicas vacías | Obtener un único conjunto con los tres años de Ausgrid y el clima |

Esta función necesita que Ausgrid llegue ya agregado a horas, en UTC y etiquetado con el final de cada hora. Esas transformaciones se definen en el cuaderno de Ausgrid.

**Lo que no se modifica.** Los valores meteorológicos se conservan tal como vienen: no se imputan, no se suavizan y no se eliminan extremos. Las marcas de la radiación tampoco se desplazan; el ajuste de $t - 30$ min solo se aplica dentro del cálculo de pvlib.

**Conjunto final.** El resultado será un único archivo en `data/processed/` con una fila por cliente y hora. Contendrá las columnas de Ausgrid de los tres años (cliente, capacidad instalada, `GC`, `CL`, `GG`, consumo total y período de origen) y las nueve variables de ERA5, con el viento ya en m/s.